[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/proyectos-integradores/03-pipeline-contenido.ipynb)

# Proyecto 3: Pipeline de contenido con IA

Genera contenido multi-formato (blog, Twitter, LinkedIn, newsletter)
a partir de un tema, con evaluación automática de calidad.

In [ ]:
import anthropic
import json
from datetime import datetime

client = anthropic.Anthropic()
PRECIOS = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00}
}
print("Cliente listo")

## 1. Prompts versionados por formato

In [ ]:
# En el proyecto real estos viven en archivos prompts/formato_v2.txt
PROMPTS = {
    "blog": """Escribe un artículo de blog completo (1200-1500 palabras) sobre: {tema}

ESTRUCTURA:
1. Título H1 con keyword
2. Introducción (150 palabras) — dato sorprendente o problema
3. 4 secciones con subtítulos H2 (250-300 palabras cada una)
4. Conclusión + CTA (100 palabras)

ESTILO: segunda persona, párrafos cortos, ejemplos concretos, tono: {tono}""",

    "twitter": """Crea un hilo de Twitter/X de 8 tweets sobre: {tema}
Audiencia: {audiencia}

- Tweet 1: gancho (máx 220 chars)
- Tweets 2-7: un punto clave por tweet con numeración (2/ 3/...)
- Tweet 8: CTA. Máx 240 chars por tweet. Emojis con moderación.""",

    "linkedin": """Escribe un post de LinkedIn (250-300 palabras) sobre: {tema}
- Primera frase: gancho potente
- Saltos de línea frecuentes (máx 2 frases por párrafo)
- Termina con pregunta para generar comentarios
- Tono: profesional y personal""",

    "newsletter": """Escribe la sección principal de una newsletter (400-500 palabras) sobre: {tema}
- Tono: cercano, como carta a un amigo
- Incluir: contexto → qué significa para el lector → consejo accionable
- Audiencia: {audiencia}"""
}

print(f"Formatos disponibles: {list(PROMPTS.keys())}")

## 2. Generador multi-formato

In [ ]:
def generar_formato(tema: str, formato: str, contexto: dict = None) -> dict:
    contexto = contexto or {}
    prompt = PROMPTS[formato].format(
        tema=tema,
        tono=contexto.get("tono", "profesional pero accesible"),
        audiencia=contexto.get("audiencia", "profesionales de tecnología")
    )
    
    # Blog → Sonnet; el resto → Haiku
    modelo = "claude-sonnet-4-6" if formato == "blog" else "claude-haiku-4-5-20251001"
    max_tok = 2000 if formato == "blog" else 700
    
    t0 = datetime.now()
    resp = client.messages.create(
        model=modelo, max_tokens=max_tok,
        messages=[{"role": "user", "content": prompt}]
    )
    duracion = (datetime.now() - t0).total_seconds()
    
    p = PRECIOS[modelo]
    coste = (resp.usage.input_tokens * p["input"] + resp.usage.output_tokens * p["output"]) / 1_000_000
    
    return {
        "formato": formato, "modelo": modelo.split("-")[1],
        "contenido": resp.content[0].text,
        "palabras": len(resp.content[0].text.split()),
        "tokens_in": resp.usage.input_tokens, "tokens_out": resp.usage.output_tokens,
        "duracion_s": round(duracion, 2), "coste_usd": round(coste, 5)
    }

# Generar un formato de prueba
TEMA = "Por qué los agentes de IA están transformando el trabajo del conocimiento en 2026"
CONTEXTO = {"tono": "experto pero accesible", "audiencia": "directores de tecnología y emprendedores"}

print(f"Generando hilo de Twitter sobre: {TEMA[:60]}...")
twitter = generar_formato(TEMA, "twitter", CONTEXTO)
print(f"✅ {twitter['palabras']} palabras | {twitter['duracion_s']}s | ${twitter['coste_usd']}")
print("\n" + twitter["contenido"])

In [ ]:
# Pipeline completo: todos los formatos
print(f"Generando pipeline completo para: {TEMA[:50]}...\n")

RESULTADOS = {}
coste_total = 0

for fmt in ["twitter", "linkedin", "newsletter"]:  # Excluimos blog para ahorrar tiempo
    print(f"  Generando {fmt}...")
    r = generar_formato(TEMA, fmt, CONTEXTO)
    RESULTADOS[fmt] = r
    coste_total += r["coste_usd"]
    print(f"  ✅ {r['palabras']} palabras | {r['duracion_s']}s | ${r['coste_usd']}")

print(f"\nCoste total (3 formatos): ${coste_total:.4f}")
print(f"Estimado con blog (Sonnet): ~${coste_total + 0.015:.3f}")

## 3. Evaluador de calidad (LLM-as-judge)

In [ ]:
def evaluar_contenido(contenido: str, formato: str, tema: str) -> dict:
    CRITERIOS = {
        "twitter": ["gancho_tweet1", "longitud_correcta", "coherencia_hilo", "potencial_engagement"],
        "linkedin": ["tono_profesional", "gancho_inicial", "legibilidad", "cta_presente"],
        "newsletter": ["cercanía_tono", "valor_para_lector", "consejo_accionable", "longitud_adecuada"],
        "blog": ["estructura_clara", "seo_friendly", "ejemplos_concretos", "cta_efectivo"]
    }
    criterios = CRITERIOS.get(formato, CRITERIOS["blog"])
    
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{"role": "user", "content": f"""Evalúa este contenido de {formato} sobre '{tema}'.
JSON sin markdown:
{{"puntuaciones": {{{', '.join(f'"{c}": 0' for c in criterios)}}},
 "nota_global": 0,
 "puntos_fuertes": ["max 2"],
 "puntos_mejora": ["max 2"],
 "listo_publicar": true}}

CONTENIDO:\n{contenido[:1500]}"""}]
    )
    evaluacion = json.loads(resp.content[0].text)
    evaluacion["formato"] = formato
    return evaluacion

print("Evaluando calidad del contenido generado:\n")
evaluaciones = {}
for fmt, datos in RESULTADOS.items():
    print(f"  Evaluando {fmt}...")
    ev = evaluar_contenido(datos["contenido"], fmt, TEMA)
    evaluaciones[fmt] = ev
    icono = "✅" if ev["listo_publicar"] else "⚠️"
    print(f"  {icono} Nota: {ev['nota_global']}/10 | Publicar: {ev['listo_publicar']}")
    print(f"     ✨ {ev['puntos_fuertes'][0] if ev['puntos_fuertes'] else 'N/A'}")
    print(f"     💡 {ev['puntos_mejora'][0] if ev['puntos_mejora'] else 'N/A'}")

In [ ]:
# Resumen del pipeline
notas = [ev["nota_global"] for ev in evaluaciones.values()]
nota_media = sum(notas) / len(notas) if notas else 0
todos_listos = all(ev["listo_publicar"] for ev in evaluaciones.values())

print("RESUMEN DEL PIPELINE")
print("=" * 40)
print(f"Formatos generados: {len(RESULTADOS)}")
print(f"Nota media de calidad: {nota_media:.1f}/10")
print(f"Listos para publicar: {'✅ Todos' if todos_listos else '⚠️ Revisar algunos'}")
print(f"Coste total: ${coste_total:.4f}")
print(f"\nEscalado a 1000 temas/mes: ${coste_total * 1000:.2f}/mes")

## 4. Versionado de prompts

Demostración del sistema de A/B testing de prompts del Bloque 21.

In [ ]:
# Comparar dos versiones del prompt de LinkedIn
LINKEDIN_V1 = "Escribe un post de LinkedIn sobre: {tema}. Sé profesional."
LINKEDIN_V2 = PROMPTS["linkedin"]  # El prompt completo con estructura

def evaluar_prompt(nombre: str, prompt_template: str, tema: str) -> dict:
    prompt = prompt_template.format(tema=tema, tono="profesional", audiencia="directivos")
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    contenido = resp.content[0].text
    
    ev = evaluar_contenido(contenido, "linkedin", tema)
    return {"version": nombre, "nota": ev["nota_global"], "listo": ev["listo_publicar"],
            "palabras": len(contenido.split())}

print("A/B test de prompts LinkedIn:\n")
res_v1 = evaluar_prompt("v1_simple", LINKEDIN_V1, TEMA)
res_v2 = evaluar_prompt("v2_estructurado", LINKEDIN_V2, TEMA)

for r in [res_v1, res_v2]:
    print(f"{r['version']:20} Nota: {r['nota']}/10 | Publicar: {r['listo']} | {r['palabras']} palabras")

ganador = res_v1 if res_v1["nota"] > res_v2["nota"] else res_v2
print(f"\n🏆 Ganador: {ganador['version']} con nota {ganador['nota']}/10")